In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# INPUT / OUTPUT
INPUT_CSV = Path("data/MASTER_VARIABLES.csv")
OUTPUT_CSV = Path("data/hazard.csv")

df = pd.read_csv(INPUT_CSV)

# ---------------------------------------------------------
# 1. DISTRICT-MONTH MEAN
# ---------------------------------------------------------
district_stats = (
    df.groupby(["dtname", "timeperiod"], as_index=False)
      .agg(heat_days_score=("heat-days-score", "mean"))
)

# ---------------------------------------------------------
# 2. MONTHLY Z-SCORE
# ---------------------------------------------------------
district_stats["heat_zscore"] = (
    district_stats.groupby("timeperiod")["heat_days_score"]
    .transform(lambda x: (x - x.mean()) / x.std(ddof=0) if x.std(ddof=0) != 0 else 0)
)

# ---------------------------------------------------------
# 3. BINNING (1–5)
# ---------------------------------------------------------
def classify(z):
    if z <= -1.5:
        return 1
    elif z <= -0.5:
        return 2
    elif z <= 0.5:
        return 3
    elif z <= 1.5:
        return 4
    else:
        return 5

district_stats["heat_hazard"] = district_stats["heat_zscore"].apply(classify)

# ---------------------------------------------------------
# 4. SAVE ONLY DISTRICT-MONTH OUTPUT (CLEAN)
# ---------------------------------------------------------
district_stats.to_csv(OUTPUT_CSV, index=False)

print(f"Saved clean hazard file: {OUTPUT_CSV}")
print(district_stats.head())


# # =============================================================================
# # 5. APPEND HAZARD TO MASTER_VARIABLES.CSV
# # =============================================================================

# master = df.copy()

# # drop old column if it exists (prevents _x/_y issues)
# if "heat_hazard" in master.columns:
#     master = master.drop(columns=["heat_hazard"])

# master = master.merge(
#     district_stats[
#         ["district", "timeperiod", "heat_hazard"]
#     ],
#     on=["district", "timeperiod"],
#     how="left",
# )

# # save back to master file
# master.to_csv(INPUT_CSV, index=False)

# print("\nUpdated master file:", INPUT_CSV)
# print("Missing hazard values:", master["heat_hazard"].isna().sum())

Saved clean hazard file: data/hazard.csv
   dtname timeperiod  heat_days_score  heat_zscore  heat_hazard
0  BAJALI    2021_01         3.166667    -0.446388            3
1  BAJALI    2021_02         9.000000    -1.400854            2
2  BAJALI    2021_03        12.666667    -1.279447            2
3  BAJALI    2021_04        14.500000    -1.441385            2
4  BAJALI    2021_05        11.500000    -1.747150            1
